# Routing BENCHMARK — Gemma 4 chess-coach (for the report)

Runs the **serious** routing ablation on Kaggle (T4, up to 12h — no Colab disconnect), over a large stratified sample of the validation set:

| condition | weights | base | isolates |
|---|---|---|---|
| **e4b-v4 adapter + harness** | trained v4 LoRA | E4B | the product |
| **e4b base + harness** | base (LoRA off) | E4B | what the **SFT weights** bought |
| **e2b adapter + harness** *(optional)* | prior E2B LoRA | E2B | the **prior production model** (E2B → E4B improvement) |

For each: confusion matrix + precision/recall/F1 + exact-name + format validity + throughput, then the deltas vs the product (adapter-vs-base = SFT weights on the same base; e4b-vs-e2b = the model-generation jump). The two E4B conditions reuse the SAME loaded model with the LoRA toggled (no 2nd load); the E2B condition has a different base, so it loads separately **after** the E4B model is freed (sequential — no OOM). No fabricated external baselines — reproducible from our own artifacts.

**The e2b condition needs its 5MB adapter** (`runs/gemma4_e2b_unified/best`, gitignored → not in the clone). Set `E2B_ADAPTER_DIR` (a Kaggle-Dataset upload) **or** `E2B_ADAPTER_REPO` (an HF push) in Cell 1; leave both blank to run the 2-condition E4B benchmark only.

**Setup:** Settings → Accelerator → **GPU T4 x2** (one is enough). Add **HF_TOKEN** under Add-ons → Secrets. Run top to bottom; the report + PNGs land in `/kaggle/working/` and render inline.

In [ ]:
# Cell 1 - config
import os
BASE_REPO = 'unsloth/gemma-4-E4B-it'   # the base the adapter was trained on
REPO_URL  = 'https://github.com/RyanDev1st/llm-and-engine.git'
BRANCH    = 'feat/chess-coach-sft'
ADAPTER_REPO = 'RyanDev1st/gemma4-chesscoach-ckpt-v4'
TAG = 'best'
WORKDIR   = '/kaggle/working/llm-and-engine'
BASE_DIR  = '/kaggle/working/gemma4_e4b'
ADAPTER_DIR = '/kaggle/working/adapter/best'
PER_SLICE   = 25          # rows/slice (0 = full val). 25 is statistically strong.
TIME_BUDGET = 1800        # 30 min/condition CAP (fast final flight; was 3h for the full ablation)
# === FAST FINAL FLIGHT (~30-40 min) — measure the POST-FIX harness ====================================
# WHY these flags: the COMPLETION eval (Cell 6.7) + the TRANSCRIPT (Cell 5) run the REAL CoachLoop, so
# they REFLECT the harness fixes (verb-coercion / generic grounding / skill-body extract-first). The
# routing cells (E4B bench, version trend, native-mode) score RAW first-action and DON'T run the loop —
# the loop-level fixes are invisible to them, and they're already measured. So the fast flight runs the
# two loop cells + keeps native ON (a fresh fair-test number for the report, capped at 30m/condition).
RUN_E4B_BENCH     = False  # Cell 4: the ~6h E4B val+stress routing benchmark (already captured)
RUN_VERSION_TREND = False  # Cell 6: per-version trend v2->v3->v4 (already measured; raw routing)
RUN_TRANSCRIPT    = True   # Cell 5: agent transcript — the ONLY proof the recipe-scaler over-ask fix landed (~3m)
RUN_NATIVE_PROBE  = True   # Cell 6.5: native-mode fair test (raw routing — won't move from loop fixes; capped ~30m)
RUN_COMPLETION    = True   # Cell 6.7: COMPLETION eval (full loop) — THE metric for the fixes + the new failing-rows log (~20m)
# === e2b: DROPPED from scope (not shipping). Both blank => skipped entirely. ==========================
E2B_ADAPTER_DIR  = ''
E2B_ADAPTER_REPO = ''
E2B_BASE_REPO    = 'unsloth/gemma-4-E2B-it'
E2B_BASE_DIR     = '/kaggle/working/gemma4_e2b'
print(f'FAST FLIGHT | budget/cond {TIME_BUDGET//60}m | e4b_bench={RUN_E4B_BENCH} '
      f'version={RUN_VERSION_TREND} transcript={RUN_TRANSCRIPT} native={RUN_NATIVE_PROBE} '
      f'completion={RUN_COMPLETION} | e2b={"on" if (E2B_ADAPTER_DIR or E2B_ADAPTER_REPO) else "off"}')

In [ ]:
# Cell 2 - clone repo (prints HEAD so the commit is visible) + deps
import subprocess, sys
env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
if not os.path.exists(WORKDIR):
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,WORKDIR], check=True, env=env)
else:
    subprocess.run(['git','-C',WORKDIR,'fetch','origin',BRANCH], check=True, env=env)
    subprocess.run(['git','-C',WORKDIR,'reset','--hard',f'origin/{BRANCH}'], check=True, env=env)
print('HEAD:', subprocess.run(['git','-C',WORKDIR,'log','-1','--oneline'], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers','accelerate','peft','bitsandbytes','sentencepiece','protobuf','python-chess'], check=True)
os.environ['PYTHONPATH'] = f'{WORKDIR}/src/llm'
import transformers, peft, bitsandbytes
print('transformers', transformers.__version__, '| peft', peft.__version__, '| bnb', bitsandbytes.__version__)

In [ ]:
# Cell 3 - download the E4B base (~9GB) + the v4 adapter (+ the tiny E2B adapter if its condition is on)
# NOTE: the E2B *base* (~10GB) is NOT downloaded here -- E4B+E2B bases together blow the ~20GB Kaggle
# disk. The last cell frees the E4B base first, then fetches the E2B base. Here we grab only the 5MB
# E2B adapter so it's ready.
from huggingface_hub import login, snapshot_download
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret('HF_TOKEN'))
except Exception:
    login(os.environ['HF_TOKEN'])
snapshot_download(repo_id=BASE_REPO, local_dir=BASE_DIR,
                  allow_patterns=['*.json','*.safetensors','*.model','*.txt','tokenizer*'])
snapshot_download(repo_id=ADAPTER_REPO, local_dir='/kaggle/working/adapter', allow_patterns=[f'{TAG}/*'])
need = ('adapter_model.safetensors', 'adapter_config.json')
assert all(os.path.exists(f'{ADAPTER_DIR}/{f}') for f in need), f'adapter not at {ADAPTER_DIR}'
print('e4b base + adapter ready:', os.listdir(ADAPTER_DIR))
E2B_ADAPTER = E2B_ADAPTER_DIR or None
if not E2B_ADAPTER and E2B_ADAPTER_REPO:          # the 5MB e2b adapter (NOT its base)
    snapshot_download(repo_id=E2B_ADAPTER_REPO, local_dir='/kaggle/working/e2b_adapter', allow_patterns=[f'{TAG}/*'])
    E2B_ADAPTER = f'/kaggle/working/e2b_adapter/{TAG}'
print('e2b adapter:', E2B_ADAPTER or 'OFF (skipping the e2b condition)')

In [ ]:
# Cell 4 - RUN THE E4B BENCHMARK: 2 conditions x both tiers (hours-safe; writes reports + PNGs)
#  - val    = in-distribution held-out rows. Conditions: e4b-v4 adapter+harness, e4b base+harness.
#  - stress = held-out WILD rows: messy/slang/typo + UNSEEN out-of-domain catalogs + decline cases.
# The E4B model loads ONCE (adapter on = product, off = base). The E2B condition runs in the LAST
# cell. Set RUN_E4B_BENCH=False in Cell 1 to skip this ~6h step if you already have its report.
import os, sys, shutil, subprocess, glob
os.environ['CHESS_HF_BASE'] = BASE_DIR
if not RUN_E4B_BENCH:
    print('RUN_E4B_BENCH=False -> skipping the E4B benchmark (using your existing report).')
else:
    env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
    for suite, ps, tb in [('val', PER_SLICE, TIME_BUDGET), ('stress', 0, 0)]:
        cmd = [sys.executable, '-m', 'llm_training.eval_benchmark', '--adapter', ADAPTER_DIR,
               '--suite', suite, '--per-slice', str(ps), '--time-budget', str(tb)]
        print('\n==== running:', ' '.join(cmd), '====\n', flush=True)
        subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    for f in glob.glob(f'{WORKDIR}/docs/findings/*-routing-benchmark*'):
        shutil.copy(f, '/kaggle/working/')
    from IPython.display import Image, Markdown, display
    for md in sorted(glob.glob('/kaggle/working/*-routing-benchmark*.md')):
        display(Markdown(open(md).read()))
    for png in sorted(glob.glob('/kaggle/working/*-routing-benchmark*.png')):
        print(png); display(Image(filename=png))

In [ ]:
# Cell 5 - CAPTURE A REAL AGENT TRANSCRIPT (for the report's "agent in action" section)
# Runs the FULL serve loop on held-out prompts over the real life-skills bundle (cooking/music/
# wellness/tax — absent from training): route by description -> load a real skill body -> call a
# real tool -> narrate the real result. Verbatim capture, not fabricated. Writes
# docs/findings/<date>-agent-transcript.md and shows it inline.
# Set RUN_TRANSCRIPT=False in Cell 1 to skip if you already captured it.
import os, sys, shutil, subprocess, glob
if not RUN_TRANSCRIPT:
    print('RUN_TRANSCRIPT=False -> skipping the transcript (using your existing capture).')
else:
    env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm', 'CHESS_HF_BASE': BASE_DIR}
    cmd = [sys.executable, '-m', 'llm_training.bench_transcript', '--adapter', ADAPTER_DIR]
    print('==== running:', ' '.join(cmd), '====\n', flush=True)
    subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    for f in glob.glob(f'{WORKDIR}/docs/findings/*-agent-transcript.md'):
        shutil.copy(f, '/kaggle/working/')
    from IPython.display import Markdown, display
    md = sorted(glob.glob('/kaggle/working/*-agent-transcript.md'))
    if md: display(Markdown(open(md[-1]).read()))

In [ ]:
# Cell 6 - MEASURED per-version routing trend (v2 -> v3 -> v4) for the report's improvement chart
# Loops each trained adapter (downloaded fresh, freed between -> no OOM), runs the SAME fast routing
# eval, and writes: the timeline chart with REAL accuracy/version + per-slice bars + the v4 per-slice
# MISS analysis (settles slice G/H) + a trend report. A missing HF repo is SKIPPED, not fatal -- so
# if your v2/v3 ids differ, edit llm_training/report/chart_data.py VERSIONS (repo/sub) and re-run.
# Set RUN_VERSION_TREND=False in Cell 1 to skip (it's raw routing -> already measured, won't move).
import os, sys, shutil, subprocess, glob
os.environ['CHESS_HF_BASE'] = BASE_DIR
if not RUN_VERSION_TREND:
    print('RUN_VERSION_TREND=False -> skipping the version trend (using your existing measurement).')
else:
    env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
    cmd = [sys.executable, '-m', 'llm_training.report.version_eval',
           '--per-slice', '15', '--time-budget', '1800', '--workdir', '/kaggle/working/adapters']
    print('==== running:', ' '.join(cmd), '====\n', flush=True)
    subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    for f in (glob.glob(f'{WORKDIR}/docs/findings/*-version-trend.md')
              + glob.glob(f'{WORKDIR}/docs/findings/report_assets/chart-*.png')
              + glob.glob(f'{WORKDIR}/docs/findings/report_assets/*-misses.jsonl')):
        shutil.copy(f, '/kaggle/working/')
    from IPython.display import Image, Markdown, display
    md = sorted(glob.glob('/kaggle/working/*-version-trend.md'))
    if md: display(Markdown(open(md[-1]).read()))
    for png in sorted(glob.glob('/kaggle/working/chart-*.png')):
        print(png); display(Image(filename=png))

In [ ]:
# Cell 6.5 - NATIVE-MODE PROOF: adapter vs base on the mode-dependent slices (the FAIR test)
# The main benchmark (Cell 4) forces FAST mode for speed -- cheap, equal handicap to both conditions,
# but it collapses the trained goal/think->action sequence on AUTO/THINK-trained slices (that's why
# slice G read 0%: the model emitted the goal keyword <skill>threats</skill> instead of chess-coach).
# Here we score each row in its TRAINED reasoning mode (--native-mode, bigger token budget) on the
# slices that depend on it. This is the honest apples-to-apples comparison and it's where the trained
# adapter pulls DECISIVELY ahead of the base on the same harness. Writes a *-val-native report.
import os, sys, shutil, subprocess, glob
os.environ['CHESS_HF_BASE'] = BASE_DIR
if not RUN_NATIVE_PROBE:
    print('RUN_NATIVE_PROBE=False -> skipping the native-mode proof.')
else:
    env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
    cmd = [sys.executable, '-m', 'llm_training.eval_benchmark', '--adapter', ADAPTER_DIR,
           '--suite', 'val', '--native-mode', '--max-new-tokens', '200', '--per-slice', '25',
           '--time-budget', str(TIME_BUDGET),
           '--slices', 'G,H,E,F,V1_S_compound_plan,V1_T_audited_plan,V1_R_compute_grounding']
    print('==== running:', ' '.join(cmd), '====\n', flush=True)
    subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    for f in glob.glob(f'{WORKDIR}/docs/findings/*-routing-benchmark-val-native*'):
        shutil.copy(f, '/kaggle/working/')
    from IPython.display import Image, Markdown, display
    for md in sorted(glob.glob('/kaggle/working/*-routing-benchmark-val-native.md')):
        display(Markdown(open(md).read()))
    for png in sorted(glob.glob('/kaggle/working/*-routing-benchmark-val-native*.png')):
        print(png); display(Image(filename=png))

In [ ]:
# Cell 6.7 - COMPLETION EVAL: the FULL serve loop, not just first-action routing (the metric the
# peer reviews said was missing). Runs the adapter over the held-out OOD STRESS suite (life-skills —
# no Stockfish needed) and scores task COMPLETION: completed / exec_ok / args_ok / grounded, plus
# the headline `recovered` = a wrong first route that the loop self-corrected to a grounded answer
# (the win strict routing records as a loss). Uses the E4B base + v4 adapter already loaded; runs
# BEFORE the e2b cell frees the base. Set RUN_COMPLETION=False in Cell 1 to skip.
import os, sys, subprocess
os.environ['CHESS_HF_BASE'] = BASE_DIR
if not RUN_COMPLETION:
    print('RUN_COMPLETION=False -> skipping the completion eval.')
else:
    env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
    cmd = [sys.executable, '-m', 'llm_training.eval_completion', '--adapter', ADAPTER_DIR,
           '--stress', '--time-budget', '1800']
    print('==== running:', ' '.join(cmd), '====\n', flush=True)
    subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)

In [ ]:
# Cell 7 - PRIOR E2B PRODUCTION MODEL (run LAST -- it frees the E4B base from disk)
# E4B base (~9GB) + E2B base (~10GB) can't both fit Kaggle's ~20GB disk. So this runs AFTER all the
# E4B work (Cells 4-6): eval_benchmark --e2b-only deletes the E4B base (--free-base), downloads the
# E2B base, then evaluates the prior E2B adapter on the SAME val rows -> a routing report tagged 'e2b'
# (verb acc / exact-name / confusion / per-slice), directly comparable to the E4B numbers above.
import os, sys, shutil, subprocess, glob
if not E2B_ADAPTER:
    print('E2B condition OFF -- set E2B_ADAPTER_REPO or E2B_ADAPTER_DIR in Cell 1 to run it.')
else:
    env = {**os.environ, 'PYTHONPATH': f'{WORKDIR}/src/llm'}
    cmd = [sys.executable, '-m', 'llm_training.eval_benchmark', '--adapter', ADAPTER_DIR,
           '--suite', 'val', '--per-slice', str(PER_SLICE), '--time-budget', str(TIME_BUDGET),
           '--e2b-only', '--e2b-adapter', E2B_ADAPTER, '--e2b-base', E2B_BASE_DIR,
           '--e2b-base-repo', E2B_BASE_REPO, '--free-base', BASE_DIR]
    print('==== running:', ' '.join(cmd), '====\n', flush=True)
    subprocess.run(cmd, cwd=f'{WORKDIR}/src/llm', env=env, check=True)
    for f in glob.glob(f'{WORKDIR}/docs/findings/*-routing-benchmark-e2b*'):
        shutil.copy(f, '/kaggle/working/')
    from IPython.display import Image, Markdown, display
    for md in sorted(glob.glob('/kaggle/working/*-routing-benchmark-e2b.md')):
        display(Markdown(open(md).read()))
    for png in sorted(glob.glob('/kaggle/working/*-routing-benchmark-e2b*.png')):
        print(png); display(Image(filename=png))